# Loading in Models

In [ ]:
import torch
import pickle
import numpy as np
import pandas as pd
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

class Model(torch.nn.Module):
    def __init__(self, input_size: int, output_size: int = 1, hidden_dim: int = 32, n_layers: int = 2, dropout: float= .5):
        super(Model, self).__init__()

        self.layers = torch.nn.ModuleList()
        self.bn_layers = torch.nn.ModuleList() # New list for BatchNorm

        # Input Layer
        # Pro-tip: bias=False because BN has its own 'beta' parameter that handles shifting
        self.layers.append(torch.nn.Linear(input_size, hidden_dim, bias=False))
        self.bn_layers.append(torch.nn.BatchNorm1d(hidden_dim))

        # Hidden Layers
        for _ in range(n_layers):
            self.layers.append(torch.nn.Linear(hidden_dim, hidden_dim, bias=False))
            self.bn_layers.append(torch.nn.BatchNorm1d(hidden_dim))

        self.dropout = torch.nn.Dropout(dropout)
        self.output_layer = torch.nn.Linear(hidden_dim, output_size)
        self.relu = torch.nn.ReLU()
        self.sigmoid = torch.nn.Sigmoid()

    def forward(self, x):
        # Use zip to iterate through both the linear and BN layers together
        for layer, bn in zip(self.layers, self.bn_layers):
            x = layer(x)
            x = bn(x)        # Normalize before activation
            x = self.relu(x)
            x = self.dropout(x)

        x = self.output_layer(x)
        return self.sigmoid(x)

NN = Model(13, n_layers = 2, hidden_dim=32, dropout = .3)

NN.load_state_dict(torch.load("mod13v2.pth"))

NN.eval()

with open('random_forest_model.pkl', 'rb') as f:
    rf = pickle.load(f)

with open('ada_boost_model.pkl', 'rb') as f:
    ad = pickle.load(f)

with open('XGBoost_model.pkl', 'rb') as f:
    xg = pickle.load(f)

with open('standard_scaler-2.pkl', 'rb') as f:
    scaler = pickle.load(f)

# First Four

In [ ]:
db26 = pd.read_csv("ff_database26.csv").drop("Unnamed: 0", axis = 1)
db26["matchup_id"] = range(4)
db26

,region,team1,team1_seed,g_team1,srs_team1,sos_team1,pts_team1,mp_team1,fg_team1,fga_team1,...,pyth_L_team2,opp_efg_team2,opp_ts_team2,opp_ftr_team2,opp_ft_fga_team2,opp_tov_pct_team2,opp_orb_pct_team2,3par_team2,opp_3par_team2,matchup_id
0,midwest,UMBC,16,32,-4.40,-11.86,2439,1290,861,1827,...,15.398164,0.469813,0.514153,0.407245,0.287596,18.634586,0.307292,0.339733,0.344676,0
1,midwest,Miami (OH),11,32,5.37,-5.49,2902,1300,1001,1909,...,15.826404,0.513998,0.545892,0.320236,0.229862,14.661476,0.315661,0.359730,0.471513,1
2,south,Prairie View A&M,16,35,-12.74,-9.93,2761,1405,932,2102,...,17.084915,0.498037,0.536499,0.315996,0.237978,14.108249,0.310981,0.389209,0.322375,2
3,west,Texas,11,32,16.17,10.30,2681,1290,905,1862,...,15.760356,0.530485,0.560619,0.365294,0.254820,15.781748,0.303082,0.439182,0.453361,3


In [ ]:
matchups = pd.read_csv("matchups26.csv").drop("Unnamed: 0", axis = 1)
teamstats26 = pd.read_csv("teamstats26.csv").drop("Unnamed: 0", axis = 1)
matchups = matchups.loc[matchups["team2"].str.contains("/"),].copy()
matchups1 = matchups.copy()
matchups1["team2"] = matchups["team2"].str.split('/', expand = True)[0]
matchups2 = matchups.copy()
matchups2["team2"] = matchups["team2"].str.split('/', expand = True)[1]
db26 = pd.concat([matchups1, matchups2])
db26["matchup_id"] = range(8)
db26

,region,team1,team1_seed,team2,team2_seed,matchup_id
8,midwest,Michigan,1,UMBC,16,0
12,midwest,Tennessee,6,Miami (OH),11,1
16,south,Florida,1,Prairie View A&M,16,2
28,west,BYU,6,Texas,11,3
8,midwest,Michigan,1,Howard,16,4
12,midwest,Tennessee,6,SMU,11,5
16,south,Florida,1,Lehigh,16,6
28,west,BYU,6,NC State,11,7


In [ ]:
seeds = {list(db26["team1"])[i] : list(db26["team1_seed"])[i] for i in range(8)} |  {list(db26["team2"])[i] : list(db26["team2_seed"])[i] for i in range(8)}
seeds

{'Michigan': 1,
 'Tennessee': 6,
 'Florida': 1,
 'BYU': 6,
 'UMBC': 16,
 'Miami (OH)': 11,
 'Prairie View A&M': 16,
 'Texas': 11,
 'Howard': 16,
 'SMU': 11,
 'Lehigh': 16,
 'NC State': 11}

In [ ]:
(list(db26["team1"]) + list(db26["team2"]))

['UMBC',
 'Miami (OH)',
 'Prairie View A&M',
 'Texas',
 'Howard',
 'SMU',
 'Lehigh',
 'NC State']

In [ ]:
seeds = {(list(db26["team1"]) + list(db26["team2"]))[i] : (list(db26["team1_seed"]) + list(db26["team2_seed"]))[i] for i in range(8)}
seeds

{'UMBC': 16,
 'Miami (OH)': 11,
 'Prairie View A&M': 16,
 'Texas': 11,
 'Howard': 16,
 'SMU': 11,
 'Lehigh': 16,
 'NC State': 11}

In [ ]:
db26 = pd.merge(pd.merge(db26, teamstats26.add_suffix("_team1"), left_on="team1", right_on="school_name_team1"),
  pd.merge(db26, teamstats26.add_suffix("_team2"), left_on="team2", right_on="school_name_team2"))

In [ ]:
scaler = StandardScaler()
db_used = db26.loc[:,list(c for c in db26.columns if any(t in c for t in {"seed","srs","sos","ORtg","DRtg","pace","efg","orb_pct","tov_pct","ft_fga", "3par"}) and "opp" not in c)].copy()
comps = list(c.split('_team1')[0] for c in db_used.columns if "_team1" in c)
db_used["seed_diff"] = db_used["team1_seed"] - db_used["team2_seed"]
for comp in comps:
    db_used[f"{comp}_diff"] = db_used[f"{comp}_team1"] - db_used[f"{comp}_team2"]
db_used = db_used.loc[:,list(c for c in db_used.columns if "diff" in c)].copy()

# adding net stats
db_used["net_efg"] = db26["efg_team1"] - db26["opp_efg_team2"]
db_used["net_tov_pct"] = db26["tov_pct_team1"] - db26["opp_tov_pct_team2"]

mirrored = -db_used.iloc[:,:-2].copy()
mirrored["net_efg"] = db26["efg_team2"] - db26["opp_efg_team1"]
mirrored["net_tov_pct"] = db26["tov_pct_team2"] - db26["opp_tov_pct_team1"]

X = np.vstack([np.array(db_used), np.array(mirrored)])

# X = np.array(db_used)
X = scaler.fit_transform(X)

In [ ]:
db_used

,seed_diff,srs_diff,sos_diff,ORtg_diff,DRtg_diff,pace_diff,efg_diff,ft_fga_diff,tov_pct_diff,orb_pct_diff,3par_diff,net_efg,net_tov_pct
0,0,0.24,-0.01,2.647788,2.795972,-3.048520,0.024812,-0.077849,-3.730482,-0.115732,0.052713,0.072332,-5.945883
1,0,-11.90,-16.15,5.660085,-6.110166,1.381643,0.054378,0.069547,-0.718329,-0.090077,0.087101,0.097841,-1.792205
2,0,-3.42,-2.89,0.739130,-4.139144,6.782206,-0.039282,0.087728,-1.617569,0.044919,-0.081883,-0.003270,0.035015
3,0,-1.85,-0.53,1.101441,1.322779,-0.325207,-0.003662,0.070305,1.971542,0.078361,-0.078817,0.019193,-2.284898


In [ ]:
db_used

,seed_diff,srs_diff,sos_diff,ORtg_diff,DRtg_diff,pace_diff,efg_diff,ft_fga_diff,tov_pct_diff,orb_pct_diff,3par_diff,net_efg,net_tov_pct
0,-15,36.88,26.72,8.241603,-2.693545,4.370152,0.038500,0.030841,1.747888,0.097746,0.025442,0.100088,1.028953
1,-5,16.71,17.48,-7.097458,-0.884499,-4.807877,-0.096221,-0.027774,1.014734,0.187036,-0.129528,0.018877,-1.480529
2,-15,40.64,23.04,13.319637,-3.271070,-1.379849,0.040527,-0.046801,-0.618850,0.151340,0.063968,0.040621,-4.261342
3,-5,4.87,2.15,-1.451229,-3.605441,1.244629,-0.003150,-0.085889,-0.455331,-0.017300,0.037367,0.037676,1.092835
4,-15,37.12,26.71,10.889391,0.102427,1.321631,0.063312,-0.047008,-1.982593,-0.017987,0.078155,0.110832,-4.197995
5,-5,4.81,1.33,-1.437373,-6.994664,-3.426234,-0.041843,0.041772,0.296405,0.096959,-0.042427,0.001619,-0.777471
6,-15,37.22,20.15,14.058767,-7.410214,5.402358,0.001245,0.040927,-2.236419,0.196259,-0.017915,0.037257,-0.583834
7,-5,3.02,1.62,-0.349788,-2.282663,0.919422,-0.006812,-0.015584,1.516211,0.061060,-0.041450,0.016043,-2.740229


In [ ]:
c1, c2, c3, c4 = [0.44440669 ,0.16464701 ,0.32866461 ,0.06228169]

xgb_probs = xg.predict_proba(X)[:,-1]
ada_probs = ad.predict_proba(X)[:,-1]
rf_probs = rf.predict_proba(X)[:,-1]
nn_probs = NN(torch.tensor(X, dtype = torch.float32)).reshape(-1).detach().numpy()

vote_pred = (c4 * xgb_probs + c2 * ada_probs +
                 c3 * rf_probs + c1 * nn_probs)

In [ ]:
db_eval = pd.DataFrame(np.vstack([db26.loc[:,["matchup_id","team1","team2","team1_seed","team2_seed"]],
           db26.loc[:,["matchup_id","team2","team1","team2_seed","team1_seed"]]]), columns = [
               "matchup_id","team1","team2","team1_seed","team2_seed"
           ])
db_eval["team1_pred"] = vote_pred
db_eval1 = db_eval[:4].reset_index().copy()
db_eval2 = db_eval[-4:].reset_index().copy()
db_eval["avg_team1_pred"] = (db_eval1["team1_pred"] + (1 - db_eval2["team1_pred"]))/2
db_eval = db_eval[:4].copy().drop("team1_pred", axis = 1)
db_eval

,matchup_id,team1,team2,team1_seed,team2_seed,avg_team1_pred
0,0,UMBC,Howard,16,16,0.579117
1,1,Miami (OH),SMU,11,11,0.244985
2,2,Prairie View A&M,Lehigh,16,16,0.331986
3,3,Texas,NC State,11,11,0.392438


In [ ]:
def log_odds(p: float) -> float:
    return abs(np.log(p/(1 - p)))

db_eval['avg_team1_pred'].apply(log_odds)

,avg_team1_pred
0,0.319148
1,1.125539
2,0.699217
3,0.437075


In [ ]:
(np.exp(1)/(1 + np.exp(1)))

np.float64(0.7310585786300049)

\begin{align}
1 &= log(\frac{p}{1-p})\\
e &= \frac{p}{1-p}\\
e - e*p &= p\\
e &= p(1 + e)\\
\frac{e}{1 + e} &= p\\
\end{align}

# Rest of field

Using UMBC, SMU, Lehigh, NC State

In [ ]:
matchups = pd.read_csv("matchups26.csv").drop("Unnamed: 0", axis = 1)
winners = {"Howard", "Miami", "Prairie View A&M", "Texas"}
for team in winners:
    matchups.loc[(matchups["team2"].str.contains("/")) &
                 (matchups["team2"].str.contains(team)),"team2"] = team if team != "Miami" else "Miami (OH)"

In [ ]:
matchups_fixed = pd.DataFrame(columns = matchups.columns)
for region in ["east","south","west","midwest"]:
    matchups_fixed = pd.concat([matchups_fixed,
                                matchups.loc[matchups["region"] == region,]])
matchups = matchups_fixed.copy()

In [ ]:
teamstats26 = pd.read_csv("teamstats26.csv").drop("Unnamed: 0", axis = 1)
db26 = pd.concat([pd.merge(matchups[["region","team1","team1_seed"]], teamstats26.add_suffix("_team1"), left_on="team1", right_on = "school_name_team1").drop("school_name_team1", axis = 1),
         pd.merge(matchups[["team2","team2_seed"]], teamstats26.add_suffix("_team2"), left_on="team2", right_on = "school_name_team2").drop("school_name_team2", axis = 1)],
          axis = 1).copy()

In [ ]:
teamstats26 = teamstats26.loc[~(teamstats26["school_name"].isin({"UMBC", "SMU", "NC State", "Lehigh"})),].copy()
db26["matchup_id"] = range(32)

seeds = {(list(db26["team1"]) + list(db26["team2"]))[i] : (list(db26["team1_seed"]) + list(db26["team2_seed"]))[i] for i in range(64)}

## specific rounds

In [ ]:
# round of 32
t1 = ["Duke", "St. John's (NY)", "Louisville", "UCLA", "Florida", "Vanderbilt",
      "VCU", "Texas A&M", "Arizona", "High Point", "Texas", "Miami (FL)",
      "Michigan", "Tennessee", "Texas Tech", "Kentucky"]
t2 = ["TCU", "Kansas", "Michigan State", "UConn", "Iowa", "Nebraska",
      "Illinois", "Houston", "Utah State", "Arkansas", "Gonzaga", "Purdue",
      "Saint Louis", "Virginia", "Alabama", "Iowa State"]

# sweet sixteen
t1 = ["Duke", "Michigan State", "Iowa", "Illinois", "Michigan", "Tennessee",
      "Arizona", "Texas"]
t2 = ["St. John's (NY)", "UConn", "Nebraska", "Houston", "Alabama",
      "Iowa State", "Arkansas", "Purdue"]

matchups32 = pd.DataFrame(
    {
     "team1" : t1, "team1_seed" : list(seeds[t] for t in t1),
     "team2" : t2, "team2_seed" : list(seeds[t] for t in t2)}
)
db26 = pd.concat([pd.merge(matchups32[["team1","team1_seed"]], teamstats26.add_suffix("_team1"), left_on="team1", right_on = "school_name_team1").drop("school_name_team1", axis = 1),
         pd.merge(matchups32[["team2","team2_seed"]], teamstats26.add_suffix("_team2"), left_on="team2", right_on = "school_name_team2").drop("school_name_team2", axis = 1)],
          axis = 1).copy()
db26

,team1,team1_seed,g_team1,srs_team1,sos_team1,pts_team1,mp_team1,fg_team1,fga_team1,fg_pct_team1,...,pyth_W_team2,pyth_L_team2,opp_efg_team2,opp_ts_team2,opp_ftr_team2,opp_ft_fga_team2,opp_tov_pct_team2,opp_orb_pct_team2,3par_team2,opp_3par_team2
0,Duke,1,34,31.55,12.41,2798,1360,973,1986,0.490,...,18.300078,15.699922,0.473580,0.510422,0.317531,0.227654,16.753440,0.302008,0.337793,0.337284
1,Michigan State,3,32,23.44,12.91,2526,1290,882,1874,0.471,...,18.475974,15.524026,0.456569,0.504673,0.401737,0.288817,15.683260,0.286796,0.403210,0.362106
2,Iowa,9,33,19.00,9.76,2482,1325,873,1777,0.491,...,17.233449,14.766551,0.478016,0.508172,0.248257,0.180161,16.633845,0.276817,0.507431,0.492761
3,Illinois,3,32,26.45,11.89,2700,1300,918,1984,0.463,...,18.735714,15.264286,0.465634,0.507800,0.390423,0.272676,18.131061,0.309137,0.407813,0.409014
4,Michigan,1,34,32.48,14.86,2952,1366,1034,2046,0.505,...,16.753478,15.246522,0.491837,0.524342,0.326531,0.227664,10.662925,0.327663,0.536528,0.350567
5,Tennessee,6,33,22.08,11.99,2622,1335,933,2017,0.463,...,18.925941,15.074059,0.495166,0.525249,0.278733,0.199248,19.752429,0.281139,0.387695,0.428034
6,Arizona,1,34,29.92,12.57,2929,1365,1039,2069,0.502,...,17.985124,16.014876,0.513569,0.545151,0.313247,0.225391,13.566478,0.312349,0.333639,0.381785
7,Texas,11,32,16.17,10.30,2681,1290,905,1862,0.486,...,18.827404,16.172596,0.522853,0.548714,0.260171,0.187343,13.795495,0.263937,0.409265,0.449523


In [ ]:
# scaler = StandardScaler()
db_used = db26.loc[:,list(c for c in db26.columns if any(t in c for t in {"seed","srs","sos","ORtg","DRtg","pace","efg","orb_pct","tov_pct","ft_fga", "3par"}) and "opp" not in c)].copy()
comps = list(c.split('_team1')[0] for c in db_used.columns if "_team1" in c)
db_used["seed_diff"] = db_used["team1_seed"] - db_used["team2_seed"]
for comp in comps:
    db_used[f"{comp}_diff"] = db_used[f"{comp}_team1"] - db_used[f"{comp}_team2"]
db_used = db_used.loc[:,list(c for c in db_used.columns if "diff" in c)].copy()

# adding net stats
db_used["net_efg"] = db26["efg_team1"] - db26["opp_efg_team2"]
db_used["net_tov_pct"] = db26["tov_pct_team1"] - db26["opp_tov_pct_team2"]

mirrored = -db_used.iloc[:,:-2].copy()
mirrored["net_efg"] = db26["efg_team2"] - db26["opp_efg_team1"]
mirrored["net_tov_pct"] = db26["tov_pct_team2"] - db26["opp_tov_pct_team1"]

X = np.vstack([np.array(db_used), np.array(mirrored)])

# X = np.array(db_used)
X = scaler.transform(X)

c1, c2, c3, c4 = [0.556 ,0.444 ,0,0]

xgb_probs = xg.predict_proba(X)[:,-1]
ada_probs = ad.predict_proba(X)[:,-1]
rf_probs = rf.predict_proba(X)[:,-1]
nn_probs = NN(torch.tensor(X, dtype = torch.float32)).reshape(-1).detach().numpy()

vote_pred = (c4 * xgb_probs + c2 * ada_probs + c3 * rf_probs + c1 * nn_probs)

In [ ]:
## for repeated simulation
# db_eval = pd.DataFrame(np.vstack([db26.loc[:,["matchup_id","team1","team2","team1_seed","team2_seed"]],
#            db26.loc[:,["matchup_id","team2","team1","team2_seed","team1_seed"]]]), columns = [
#                "matchup_id","team1","team2","team1_seed","team2_seed"
#            ])
# db_eval = pd.concat([db26["region"], db_eval], axis = 1)

## for round specific matchups
db_eval = pd.DataFrame(np.vstack([db26.loc[:,["team1","team2","team1_seed","team2_seed"]],
           db26.loc[:,["team2","team1","team2_seed","team1_seed"]]]), columns = [
               "team1","team2","team1_seed","team2_seed"
           ])
db_eval["team1_pred"] = vote_pred
db_eval1 = db_eval[:len(db_eval)//2].reset_index().copy()
db_eval2 = db_eval[-len(db_eval)//2:].reset_index().copy()
db_eval["avg_team1_pred"] = (db_eval1["team1_pred"] + (1 - db_eval2["team1_pred"]))/2
db_eval = db_eval[:len(db_eval)//2].copy().drop("team1_pred", axis = 1)

In [ ]:
db_eval

,team1,team2,team1_seed,team2_seed,avg_team1_pred
0,Duke,St. John's (NY),1,5,0.631113
1,Michigan State,UConn,3,2,0.569174
2,Iowa,Nebraska,9,4,0.393048
3,Illinois,Houston,3,2,0.497598
4,Michigan,Alabama,1,4,0.631096
5,Tennessee,Iowa State,6,2,0.344800
6,Arizona,Arkansas,1,4,0.607474
7,Texas,Purdue,11,2,0.335361


In [ ]:
db_eval.iloc[:,].to_csv("R64_preds.csv")

# Simming

In [ ]:
class Bracket:

    def __init__(self, round_64: list[tuple[str,str]]):

        self.rounds: dict[int, list[tuple[str, str]]] = dict()
        self.rounds[64] = round_64
        for round in list(2**i for i in range(5,0,-1)):
            self.rounds[round] = list()

In [ ]:
def MC_MM(finishes: "pd.DataFrame", bracket) -> "pd.DataFrame":

    needed_cols: list[str] = ["school_name","srs","sos","ORtg","DRtg","pace","efg", "opp_efg",
                              "orb_pct","tov_pct", "opp_tov_pct", "ft_fga", "3par"]

    # scaler = StandardScaler()
    db_used = db26.loc[:,list(c for c in db26.columns if any(t in c for t in {"seed","srs","sos","ORtg","DRtg","pace","efg","orb_pct","tov_pct","ft_fga", "3par"}) and "opp" not in c)].copy()
    comps = list(c.split('_team1')[0] for c in db_used.columns if "_team1" in c)
    db_used["seed_diff"] = db_used["team1_seed"] - db_used["team2_seed"]
    for comp in comps:
        db_used[f"{comp}_diff"] = db_used[f"{comp}_team1"] - db_used[f"{comp}_team2"]
    db_used = db_used.loc[:,list(c for c in db_used.columns if "diff" in c)].copy()

    # adding net stats
    db_used["net_efg"] = db26["efg_team1"] - db26["opp_efg_team2"]
    db_used["net_tov_pct"] = db26["tov_pct_team1"] - db26["opp_tov_pct_team2"]

    mirrored = -db_used.iloc[:,:-2].copy()
    mirrored["net_efg"] = db26["efg_team2"] - db26["opp_efg_team1"]
    mirrored["net_tov_pct"] = db26["tov_pct_team2"] - db26["opp_tov_pct_team1"]

    X = np.vstack([np.array(db_used), np.array(mirrored)])

    # X = np.array(db_used)
    X = scaler.transform(X)
    db_next = db26.copy()

    rounds: list[int] = list(2**i for i in range(6,0,-1))
    # pd.DataFrame([64]*32, columns = ["round"])
    results: "pd.DataFrame" = pd.DataFrame(columns = ["round",'team1', 'team2',
      'team1_seed', 'team2_seed', 'team1_pred', 'avg_pred','winner'])
    for round in rounds:

        c1, c2, c3, c4 = [.556, .444, 0, 0]

        xgb_probs = xg.predict_proba(X)[:,-1]
        ada_probs = ad.predict_proba(X)[:,-1]
        rf_probs = rf.predict_proba(X)[:,-1]
        nn_probs = NN(torch.tensor(X, dtype = torch.float32)).reshape(-1).detach().numpy()

        vote_pred = (c4 * xgb_probs + c2 * ada_probs +
                        c3 * rf_probs + c1 * nn_probs)

        db_eval = pd.DataFrame(np.vstack([db_next.loc[:,["team1","team2","team1_seed","team2_seed"]],
                  db_next.loc[:,["team2","team1","team2_seed","team1_seed"]]]), columns = [
                      "team1","team2","team1_seed","team2_seed"
                  ])
        db_eval["team1_pred"] = vote_pred
        # db_eval.loc[(db_eval["team1_seed"] == 10) | (db_eval["team2_seed"] == 10),]
        # db_pred["pred_win"] = 0
        # db_eval.loc[(vote_pred >= 0.46697903412679714),"pred_win"] = 1

        db_eval1 = db_eval[:(round//2)].reset_index().copy()
        db_eval2 = db_eval[-(round//2):].reset_index().copy()
        db_eval["avg_pred"] = (db_eval1["team1_pred"] + (1 - db_eval2["team1_pred"]))/2
        db_eval = db_eval[:(round//2)].copy()
        db_eval["winner"] = ""
        for i in range(len(db_eval)):
            t = np.random.random()
            if t <= db_eval.loc[i, "avg_pred"]:
                db_eval.loc[i, "winner"] = db_eval.loc[i, "team1"]
            else:
                db_eval.loc[i, "winner"] = db_eval.loc[i, "team2"]
            # list(zip(db25["team1"], db25["team2"]))
        # bracket.rounds[round//2] = list((db_eval["winner"].values[::2][i],
        #                                 db_eval["winner"].values[1::2][i]) for i in range(round//4))
        bracket.rounds[round//2] = list(db_eval["winner"].values)
        if len(results) > 1:
            results = pd.concat([results,
                pd.concat([
                pd.DataFrame([round]*(round//2), columns = ["round"]),
                db_eval
                ], axis = 1)], axis = 0)
        else:
            results = pd.concat([
                pd.DataFrame([round]*(round//2), columns = ["round"]),
                db_eval
                ], axis = 1)
        if round != 2:
            team1: list[str] = list()
            team2: list[str] = list()
            for i in range(0,len(db_eval),2):
                t1, t2 = list(db_eval["winner"])[i:(i +2)]
                team1.append(t1)
                team2.append(t2)
                # bracket.rounds[round].append((t1,t2))

            db_next = pd.DataFrame(columns = ["team1","team2"])
            db_next["team1"] = team1
            db_next["team2"] = team2

            # print('here')
            db_next = pd.merge(
                pd.merge(db_next, teamstats26.loc[:,needed_cols].add_suffix("_team1"), left_on = "team1", right_on = "school_name_team1"),
                pd.merge(db_next, teamstats26.loc[:,needed_cols].add_suffix("_team2"), left_on = "team2", right_on = "school_name_team2"))
            db_next["team1_seed"] = 0
            db_next["team2_seed"] = 0
            for i in range(len(db_next)):
                db_next.loc[i, "team1_seed"] = seeds[db_next.loc[i, "team1"]]
                db_next.loc[i, "team2_seed"] = seeds[db_next.loc[i, "team2"]]
            db_next = db_next.drop(["school_name_team1", "school_name_team2"], axis = 1)

            # db_used = db25.loc[:,list(c for c in db25.columns if any(t in c for t in {"seed","SRS","SOS","off_rtg","def_rtg","pace","eFG","ORB_pct","TOV_pct","FT_FGA", "3PAr"}) and "opp" not in c)].copy()
            db_used = db_next.copy()
            comps = list(c.split('_team1')[0] for c in db_used.columns if "_team1" in c)
            db_used["seed_diff"] = db_used["team1_seed"] - db_used["team2_seed"]
            for comp in comps:
                db_used[f"{comp}_diff"] = db_used[f"{comp}_team1"] - db_used[f"{comp}_team2"]
            db_used = db_used.loc[:,list(c for c in db_used.columns if "diff" in c and "opp" not in c)].copy()

            # adding net stats
            db_used["net_efg"] = db_next["efg_team1"] - db_next["opp_efg_team2"]
            db_used["net_tov_pct"] = db_next["tov_pct_team1"] - db_next["opp_tov_pct_team2"]

            mirrored = -db_used.iloc[:,:-2].copy()
            mirrored["net_efg"] = db_next["efg_team2"] - db_next["opp_efg_team1"]
            mirrored["net_tov_pct"] = db_next["tov_pct_team2"] - db_next["opp_tov_pct_team1"]

            X = np.vstack([np.array(db_used), np.array(mirrored)])

            X = scaler.transform(X)

    for team in teamstats26["school_name"]:
        finish = results.loc[(results["team1"] == team) |
        (results["team2"] == team),"round"].min()
        # finishes[team] = finish
        if finish == 2:
            # bracket.rounds[2].append(team)
            if team == results.loc[results["round"] == 2,"winner"].values[0]:
                finish = 1
                bracket.rounds[1] = team
            # else:
            #     bracket.rounds[2].append(team)
        finishes.loc[finishes["team"] == team,str(finish)] += 1


    return finishes, bracket

In [ ]:
bracket = Bracket(round_64 = [(db26["team1"][i], db26["team2"][i]) for i in range(32)])
finishes = pd.DataFrame(columns = ["team"] + [str(2**i) for i in range(6,-1,-1)])
finishes["team"] = teamstats26["school_name"]
for col in finishes.columns[1:]:
    finishes[col] = [0]*64
finishes["seed"] = list(seeds[team] for team in finishes["team"].values)
n: int = 1
i: int = 0
start = time.time()

finishes, bracket = MC_MM(finishes, bracket)

brackets_table: "pd.DataFrame" = pd.DataFrame(columns = ["id","round","seed1","team1","seed2","team2","winner"])
start = time.time()
for id in range(1):
    for round in list(2**i for i in range(6,0,-1)):
        if round == 64:
            for matchup in bracket.rounds[64]:
                t1, t2 = matchup
                winner = t1 if t1 in bracket.rounds[32] else t2
                brackets_table.loc[len(brackets_table)] = [id, 64, seeds[t1], t1, seeds[t2], t2, winner]
        else:
            for matchup in zip(bracket.rounds[round][::2], bracket.rounds[round][1::2]):
                t1, t2 = matchup
                winner = t1 if t1 in bracket.rounds[round//2] else t2
                brackets_table.loc[len(brackets_table)] = [id, round, seeds[t1], t1, seeds[t2], t2, winner]

In [ ]:
import time

In [ ]:
finishes = pd.DataFrame(columns = ["team"] + [str(2**i) for i in range(6,-1,-1)])
finishes["team"] = teamstats26["school_name"]
for col in finishes.columns[1:]:
    finishes[col] = [0]*64
finishes["seed"] = list(seeds[team] for team in finishes["team"].values)
n: int = 10000
i: int = 0
start = time.time()

brackets = list(Bracket(round_64 = [(db26["team1"][i], db26["team2"][i]) for i in range(32)]) for _ in range(n))
for brack in brackets:
    finishes, brack = MC_MM(finishes, brack)
    i += 1
    if (i + 1) % 500 == 0:
        print(f"Bracket {i}: {(time.time() - start)/60}")

Bracket 499: 4.93684899409612
Bracket 999: 9.796786638100942
Bracket 1499: 14.63274538119634
Bracket 1999: 19.4262779990832
Bracket 2499: 24.284495039780936
Bracket 2999: 29.06810760498047
Bracket 3499: 33.866116269429526
Bracket 3999: 38.586540516217546
Bracket 4499: 43.3236301779747
Bracket 4999: 48.06802263657252
Bracket 5499: 52.826483233769736
Bracket 5999: 57.66091501712799
Bracket 6499: 62.537595856189725
Bracket 6999: 67.32829364935557
Bracket 7499: 72.1398713986079
Bracket 7999: 77.13016480207443
Bracket 8499: 82.00306512912114
Bracket 8999: 86.83827810287475
Bracket 9499: 91.62778259913127
Bracket 9999: 96.46333230336508


In [ ]:
from collections import Counter

def bracket_signature(bracket):

    sig = []

    # rounds = {round:brackets[0].rounds[round] for round in [16,8,4,2,1]}

    rounds = {round:bracket.rounds[round] for round in [16,8,4,2,1]}
    for r in sorted(rounds.keys(), reverse = True):
    # for r in sorted(bracket.rounds.keys(), reverse=True):
        games = bracket.rounds[r]

        if isinstance(games, list):
            sig.append(tuple(games))
        else:  # championship winner
            sig.append(games)

    return tuple(sig)

counter = Counter()

for bracket in brackets:
    sig = bracket_signature(bracket)
    counter[sig] += 1

In [ ]:
{sig:counter[sig] for sig in counter.keys() if counter[sig] >= 2}

{(('Duke',
   'Kansas',
   'Louisville',
   'UConn',
   'Michigan',
   'Texas Tech',
   'Tennessee',
   'Iowa State',
   'Florida',
   'Vanderbilt',
   'Illinois',
   "Saint Mary's",
   'Arizona',
   'Wisconsin',
   'Gonzaga',
   'Purdue'),
  ('Duke',
   'UConn',
   'Michigan',
   'Iowa State',
   'Florida',
   'Illinois',
   'Arizona',
   'Purdue'),
  ('Duke', 'Michigan', 'Illinois', 'Purdue'),
  ('Michigan', 'Purdue'),
  'Michigan'): 2}

In [ ]:
def matchup_investigator(brackets: list["Bracket"], team1: str, team2: str, round: int) -> dict[str, float]:

    brackets = list(b for b in brackets if team1 in b.rounds[round] and team2 in b.rounds[round])

    team1_wins = list(b for b in brackets if team1 in b.rounds[round//2])

    return {team1: len(team1_wins), team2: len(brackets) - len(team1_wins)}

In [ ]:
# brackets[777].rounds[32]
matchup_investigator(brackets, "St. John's (NY)", "Kansas", 32)

{"St. John's (NY)": 2932, 'Kansas': 2907}

In [ ]:
# brackets[0].rounds

In [ ]:
t = [1,2,3,4]
t = zip(t,t)
# for i in range(len(t)//2):
#     print(t[(2*i):(2*i) + 2])
for i,j in t:
    print(i, j)

1 1
2 2
3 3
4 4


In [ ]:
brackets_table: "pd.DataFrame" = pd.DataFrame(columns = ["id","round","seed1","team1","seed2","team2","winner"])
start = time.time()
for id in range(10000):
    for round in list(2**i for i in range(6,0,-1)):
        if round == 64:
            for matchup in brackets[id].rounds[64]:
                t1, t2 = matchup
                winner = t1 if t1 in brackets[id].rounds[32] else t2
                brackets_table.loc[len(brackets_table)] = [id, 64, seeds[t1], t1, seeds[t2], t2, winner]
        else:
            for matchup in zip(brackets[id].rounds[round][::2], brackets[id].rounds[round][1::2]):
                t1, t2 = matchup
                winner = t1 if t1 in brackets[id].rounds[round//2] else t2
                brackets_table.loc[len(brackets_table)] = [id, round, seeds[t1], t1, seeds[t2], t2, winner]
    if (id + 1) % 1000 == 0:
        print(f"Bracket {id + 1}: {(time.time() - start)/60}")
        brackets_table.to_csv(f"brackets{id + 1}.csv")
        # files.download(f"brackets{id + 1}.csv")
        brackets_table: "pd.DataFrame" = pd.DataFrame(columns = ["id","round","seed1","team1","seed2","team2","winner"])

Bracket 1000: 3.1893989284833273
Bracket 2000: 6.334137193361918
Bracket 3000: 9.558085747559865
Bracket 4000: 12.666831930478414
Bracket 5000: 15.70684806505839
Bracket 6000: 18.75829083919525
Bracket 7000: 21.857530935605368
Bracket 8000: 24.897450387477875
Bracket 9000: 27.981577010949454
Bracket 10000: 31.018313745657604


In [ ]:
brackets_table: "pd.DataFrame" = pd.DataFrame(columns = ["id","round","seed1","team1","seed2","team2","winner"])
for b in range(1000, 11000, 1000):
    brackets_table = pd.concat([brackets_table,
                                pd.read_csv(f"brackets{b}.csv").drop("Unnamed: 0", axis = 1)])

In [ ]:
from google.colab import files
brackets_table.to_csv("brackets.csv")
files.download("brackets.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
brackets_table.to_csv("brackets1000.csv")
files.download("brackets1000.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
finishes_analysis = finishes.copy()
finishes_analysis["32_prob"] = 1 - finishes_analysis["64"]/n
finishes_analysis["16_prob"] = 1 - (finishes_analysis["64"] + finishes_analysis["32"])/n
finishes_analysis["8_prob"] = 1 - (finishes_analysis["64"] + finishes_analysis["32"] + finishes_analysis["16"])/n
finishes_analysis["4_prob"] = 1 - (finishes_analysis["64"] + finishes_analysis["32"] + finishes_analysis["16"] + finishes_analysis["8"])/n
finishes_analysis["2_prob"] = 1 - (finishes_analysis["64"] + finishes_analysis["32"] + finishes_analysis["16"] + finishes_analysis["8"] + finishes_analysis["4"])/n
finishes_analysis["1_prob"] = finishes["1"]/n

In [ ]:
team = pd.DataFrame(columns = ["team"])
for i in range(32):
    team.loc[len(team.index)] = list(db26["team1"])[i]
    team.loc[len(team.index)] = list(db26["team2"])[i]
finishes_analysis = pd.merge(team, finishes_analysis).copy()

In [ ]:
# finishes_analysis.loc[finishes_analysis["team"] == "SMU",]
# finishes_analysis #.loc[(finishes_analysis["seed"] == 1) | (finishes_analysis["seed"] == 16),]
finishes_analysis.sort_values("2_prob", ascending = 0).head(10)

,team,64,32,16,8,4,2,1,seed,32_prob,16_prob,8_prob,4_prob,2_prob,1_prob
0,Duke,559,1386,2039,1497,1673,1104,1742,1,0.9441,0.8055,0.6016,0.4519,0.2846,0.1742
48,Michigan,655,2160,2292,1762,1143,789,1199,1,0.9345,0.7185,0.4893,0.3131,0.1988,0.1199
32,Arizona,1206,1280,2690,1748,1180,804,1092,1,0.8794,0.7514,0.4824,0.3076,0.1896,0.1092
62,Iowa State,769,2515,2142,2145,1154,607,668,2,0.9231,0.6716,0.4574,0.2429,0.1275,0.0668
16,Florida,547,2987,2564,1598,1109,560,635,1,0.9453,0.6466,0.3902,0.2304,0.1195,0.0635
46,Purdue,784,3022,2220,1757,1036,590,591,2,0.9216,0.6194,0.3974,0.2217,0.1181,0.0591
26,Illinois,1387,2326,2742,1493,934,531,587,3,0.8613,0.6287,0.3545,0.2052,0.1118,0.0587
30,Houston,1760,2431,2320,1442,972,496,579,2,0.8240,0.5809,0.3489,0.2047,0.1075,0.0579
42,Gonzaga,1736,2980,2751,1203,695,346,289,3,0.8264,0.5284,0.2533,0.1330,0.0635,0.0289
20,Vanderbilt,870,3776,2645,1535,624,310,240,5,0.9130,0.5354,0.2709,0.1174,0.0550,0.0240


In [ ]:
from google.colab import files
finishes_analysis.to_csv("26finishes10k.csv")
files.download("26finishes10k.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Nooticing

In [ ]:
brackets = pd.read_csv("brackets.csv").drop("Unnamed: 0", axis = 1)
brackets.head()

,id,round,seed1,team1,seed2,team2,winner
0,0,64,1,Duke,16,Siena,Duke
1,0,64,8,Ohio State,9,TCU,Ohio State
2,0,64,5,St. John's (NY),12,Northern Iowa,St. John's (NY)
3,0,64,4,Kansas,13,California Baptist,Kansas
4,0,64,6,Louisville,11,South Florida,Louisville


In [ ]:
def matchup_investigator(b: "pd.DataFrame", team1: str, team2: str, research: "pd.DataFrame"):

    b = b.loc[(b["team1"] == team1) & (b["team2"] == team2),]
    n = len(b)
    round = b["round"].unique()[0]
    s1, s2 = b[["seed1", "seed2"]].drop_duplicates().values[0]

    team1_wins = len(b.loc[b["winner"] == team1,])

    research.loc[len(research.index)] = [round, s1, team1, team1_wins,
                                         s2, team2, n - team1_wins, team1_wins/n]

    return research

In [ ]:
research: "pd.DataFrame" = pd.DataFrame(columns = ["round", "team1_seed", "team1", "team1_wins",
                                                   "team2_seed","team2", "team2_wins", "team1_pred"])

for t1, t2 in brackets.loc[(brackets["seed1"] == 1) & ((brackets["seed2"] == 8) | (brackets["seed2"] == 9)) & (brackets["round"] == 32),["team1", "team2"]].drop_duplicates().values:
    research = matchup_investigator(brackets, t1, t2, research)

In [ ]:
research

,round,team1_seed,team1,team1_wins,team2_seed,team2,team2_wins,team1_pred
0,32,1,Duke,4604,8,Ohio State,831,0.847102
1,32,1,Florida,2888,8,Clemson,733,0.797570
2,32,1,Arizona,3842,9,Utah State,996,0.794130
3,32,1,Michigan,4093,8,Georgia,688,0.856097
4,32,1,Florida,3451,9,Iowa,1499,0.697172
5,32,1,Arizona,3232,8,Villanova,924,0.777671
6,32,1,Michigan,3741,9,Saint Louis,577,0.866373
7,32,1,Duke,3243,9,TCU,419,0.885582


In [ ]:
# brackets_table.loc[(brackets_table["id"] == 0) & (brackets_table["round"] == 64),]

In [ ]:
finishes_analysis.loc[(finishes_analysis["team"] == "Kentucky") |
 (finishes_analysis["team"] == "Santa Clara"),]

,team,64,32,16,8,4,2,1,seed,32_prob,16_prob,8_prob,4_prob,2_prob,1_prob
28,Kentucky,3480,4068,1135,985,242,69,21,7,0.652,0.2452,0.1317,0.0332,0.009,0.0021
29,Santa Clara,6520,2462,717,235,56,9,1,10,0.348,0.1018,0.0301,0.0066,0.001,0.0001


## R64

In [ ]:
R64 = pd.read_csv("R64_preds.csv").drop("Unnamed: 0", axis = 1)
R64

,region,matchup_id,team1,team2,team1_seed,team2_seed,avg_team1_pred
0,east,0,Duke,Siena,1,16,0.910370
1,east,1,Ohio State,TCU,8,9,0.607269
2,east,2,St. John's (NY),Northern Iowa,5,12,0.682788
3,east,3,Kansas,California Baptist,4,13,0.853252
4,east,4,Louisville,South Florida,6,11,0.725222
5,east,5,Michigan State,North Dakota State,3,14,0.784981
6,east,6,UCLA,UCF,7,10,0.628855
7,east,7,UConn,Furman,2,15,0.849566
8,midwest,8,Michigan,UMBC,1,16,0.908622
9,midwest,9,Georgia,Saint Louis,8,9,0.527100


## finishes

In [ ]:
brackets = brackets_table.copy()

In [ ]:
research: "pd.DataFrame" = pd.DataFrame(columns = ["round", "team1_seed", "team1", "team1_wins",
                                                   "team2_seed","team2", "team2_wins", "team1_pred"])

for t1, t2 in brackets.loc[(brackets["seed1"] == 1) & (brackets["seed2"] == 2),["team1", "team2"]].drop_duplicates().values:
    research = matchup_investigator(brackets, t1, t2, research)

In [ ]:
research

,round,team1_seed,team1,team1_wins,team2_seed,team2,team2_wins,team1_pred
0,8,1,Arizona,1316,2,Purdue,1329,0.497543
1,2,1,Duke,308,2,Purdue,264,0.538462
2,8,1,Florida,903,2,Houston,589,0.605228
3,4,1,Duke,730,2,Houston,179,0.803080
4,8,1,Michigan,2194,2,Iowa State,599,0.785535
5,8,1,Duke,1600,2,UConn,258,0.861141
6,4,1,Arizona,428,2,Iowa State,182,0.701639
7,2,1,Duke,269,2,Iowa State,49,0.845912
8,2,1,Florida,80,2,Iowa State,51,0.610687
9,2,1,Florida,101,2,Purdue,91,0.526042


In [ ]:
# brackets.loc[(brackets["team1"] == "Duke") & (brackets["round"] == 2) & (brackets["winner"] == "Duke"),]

In [ ]:
finishes = pd.read_csv("26finishes10k.csv").drop("Unnamed: 0", axis = 1)
finishes.sort_values("8_prob",ascending = 0).head(12)
# finishes.loc[(finishes["seed"] == 1) | (finishes["seed"] == 2),]

,team,64,32,16,8,4,2,1,seed,32_prob,16_prob,8_prob,4_prob,2_prob,1_prob
0,Duke,878,1357,1289,1070,1239,1828,2339,1,0.9122,0.7765,0.6476,0.5406,0.4167,0.2339
48,Michigan,925,1276,1554,1091,1034,966,3154,1,0.9075,0.7799,0.6245,0.5154,0.4120,0.3154
32,Arizona,966,1879,1578,1993,1957,754,873,1,0.9034,0.7155,0.5577,0.3584,0.1627,0.0873
16,Florida,1362,2095,1817,1600,1505,968,653,1,0.8638,0.6543,0.4726,0.3126,0.1621,0.0653
46,Purdue,1809,1773,1729,1812,1619,610,648,2,0.8191,0.6418,0.4689,0.2877,0.1258,0.0648
62,Iowa State,1159,2374,2031,2777,878,463,318,2,0.8841,0.6467,0.4436,0.1659,0.0781,0.0318
26,Illinois,2047,1679,2039,1826,1319,622,468,3,0.7953,0.6274,0.4235,0.2409,0.1090,0.0468
30,Houston,2156,2212,2510,1436,960,479,247,2,0.7844,0.5632,0.3122,0.1686,0.0726,0.0247
10,Michigan State,2126,3373,1647,1822,589,306,137,3,0.7874,0.4501,0.2854,0.1032,0.0443,0.0137
14,UConn,1413,2696,3041,2015,558,201,76,2,0.8587,0.5891,0.2850,0.0835,0.0277,0.0076
